# 📖 Notebook 2: Distributed Task Execution

In Notebook 1, we built a priority queue. Now we need **workers** to pull jobs from that queue and execute them reliably. The hard part: what happens when a worker crashes mid-job?

## Learning Objectives

- Build a worker pool that pulls jobs from Redis
- Implement visibility timeouts so crashed jobs get retried
- Add exponential backoff for failed jobs
- Understand at-least-once delivery and idempotency

## 🛠️ Setup

Start the infrastructure first:

```bash
cd system-designs/job-scheduler
docker-compose up -d
```

### Visualization Tools

- **Adminer** (PostgreSQL GUI): http://localhost:8080  
  Login: System `PostgreSQL`, Server `postgres`, User `demo`, Password `demo`, Database `job_scheduler`
- **RedisInsight** (Redis GUI): http://localhost:5540  
  Click "Add Redis Database" → Host `redis`, Port `6379`

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import redis
import json
import time
import uuid
import random
import threading
from datetime import datetime, timedelta

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "job_scheduler",
    "user": "demo",
    "password": "demo"
}

REDIS_CONFIG = {
    "host": "localhost",
    "port": 6379,
    "decode_responses": True
}

def get_db():
    return psycopg2.connect(**DB_CONFIG)

def get_redis():
    return redis.Redis(**REDIS_CONFIG)

r = get_redis()

try:
    conn = get_db()
    conn.close()
    print("✅ Connected to PostgreSQL")
except Exception as e:
    print(f"❌ PostgreSQL failed: {e}")

try:
    r.ping()
    print("✅ Connected to Redis")
except Exception as e:
    print(f"❌ Redis failed: {e}")

## 🤔 The Problem: Workers Can Crash

Imagine Worker A pulls a job from the queue and starts processing it. Midway through, the worker crashes (out of memory, network failure, hardware issue). What happens to that job?

```
Worker A pulls "send_email" from queue
   ↓
Worker A starts processing...
   ↓
💥 Worker A CRASHES
   ↓
The job is gone from the queue but never completed!
```

**The fix: visibility timeouts.**

When a worker takes a job, we don't delete it — we *hide* it for a limited time. If the worker doesn't confirm completion before the timeout, the job becomes visible again for another worker.

```
Worker A takes job → job hidden for 30 seconds
  │
  ├─ Worker A finishes → deletes job ✅
  │
  └─ Worker A crashes → 30s passes → job reappears → Worker B picks it up ✅
```

In [ ]:
# Visibility timeout implementation using Redis
#
# Strategy:
# - Main queue: ZSET with scheduled_at as score
# - Processing set: ZSET with visibility_timeout as score
# - When a worker takes a job, move it from main → processing
# - If worker finishes, remove from processing
# - A reaper checks processing set for expired timeouts

QUEUE_KEY = "worker_demo:queue"
PROCESSING_KEY = "worker_demo:processing"
VISIBILITY_TIMEOUT = 30  # seconds

# Clean up
r.delete(QUEUE_KEY, PROCESSING_KEY)

def enqueue(execution_id: str, task_id: str, params: dict, scheduled_at: float = None):
    """Add a job to the queue."""
    if scheduled_at is None:
        scheduled_at = time.time()
    member = json.dumps({
        "execution_id": execution_id,
        "task_id": task_id,
        "parameters": params,
        "attempt": 0
    })
    r.zadd(QUEUE_KEY, {member: scheduled_at})

def take_job(worker_id: str) -> dict | None:
    """Atomically take the next ready job and start the visibility timeout.
    
    The job moves from the main queue to the processing set.
    If we don't ACK it within VISIBILITY_TIMEOUT seconds, it goes back.
    """
    now = time.time()
    
    # Get the next ready job
    results = r.zrangebyscore(QUEUE_KEY, "-inf", now, start=0, num=1)
    if not results:
        return None
    
    member = results[0]
    
    # Try to remove it (atomic — only one worker wins)
    if not r.zrem(QUEUE_KEY, member):
        return None  # another worker got it first
    
    # Add to processing set with visibility timeout as score
    job = json.loads(member)
    job["worker_id"] = worker_id
    job["taken_at"] = now
    processing_member = json.dumps(job)
    timeout_score = now + VISIBILITY_TIMEOUT
    r.zadd(PROCESSING_KEY, {processing_member: timeout_score})
    
    return job

def ack_job(job: dict):
    """Acknowledge successful completion — remove from processing set."""
    member = json.dumps(job)
    r.zrem(PROCESSING_KEY, member)

def requeue_expired():
    """Find jobs in processing set whose visibility timeout has expired.
    
    These are jobs where the worker crashed or took too long.
    Move them back to the main queue for another worker to pick up.
    """
    now = time.time()
    expired = r.zrangebyscore(PROCESSING_KEY, "-inf", now)
    
    requeued = 0
    for member in expired:
        if r.zrem(PROCESSING_KEY, member):
            job = json.loads(member)
            job["attempt"] = job.get("attempt", 0) + 1
            # Remove worker tracking fields before re-enqueuing
            job.pop("worker_id", None)
            job.pop("taken_at", None)
            new_member = json.dumps(job)
            r.zadd(QUEUE_KEY, {new_member: now})  # ready immediately
            requeued += 1
    
    return requeued

print("✅ Visibility timeout functions defined")
print(f"   Timeout: {VISIBILITY_TIMEOUT}s")
print("   take_job(worker_id) → moves job to processing set")
print("   ack_job(job) → removes from processing set")
print("   requeue_expired() → returns timed-out jobs to queue")

In [ ]:
# Demo: visibility timeout in action

# Add some jobs
for i in range(5):
    enqueue(f"exec-{i+1}", "send_email", {"to": f"user{i+1}@example.com"})

print(f"📋 Queue: {r.zcard(QUEUE_KEY)} jobs | Processing: {r.zcard(PROCESSING_KEY)} jobs")
print()

# Worker A takes a job
job_a = take_job("worker-A")
print(f"👷 Worker A took: {job_a['execution_id']}")
print(f"📋 Queue: {r.zcard(QUEUE_KEY)} jobs | Processing: {r.zcard(PROCESSING_KEY)} jobs")
print()

# Worker B takes a job
job_b = take_job("worker-B")
print(f"👷 Worker B took: {job_b['execution_id']}")
print(f"📋 Queue: {r.zcard(QUEUE_KEY)} jobs | Processing: {r.zcard(PROCESSING_KEY)} jobs")
print()

# Worker A finishes successfully
ack_job(job_a)
print(f"✅ Worker A completed {job_a['execution_id']} and ACKed")
print(f"📋 Queue: {r.zcard(QUEUE_KEY)} jobs | Processing: {r.zcard(PROCESSING_KEY)} jobs")
print()

# Worker B "crashes" — we never call ack_job(job_b)
print(f"💥 Worker B CRASHED while processing {job_b['execution_id']}!")
print(f"   Job is stuck in processing set with timeout score")
print()

# Simulate timeout by directly manipulating the score
# (In production the reaper runs on a timer)
member = json.dumps(job_b)
r.zadd(PROCESSING_KEY, {member: time.time() - 1})  # force expiry

requeued = requeue_expired()
print(f"🔄 Reaper found {requeued} expired job(s) and requeued them")
print(f"📋 Queue: {r.zcard(QUEUE_KEY)} jobs | Processing: {r.zcard(PROCESSING_KEY)} jobs")
print()
print("💡 The crashed worker's job is back in the queue for another worker!")

## 🔁 Exponential Backoff for Retries

When a job fails (e.g. an API returns a 500 error), we don't want to retry immediately — the upstream service might still be down. Instead, we wait longer between each retry:

```
Attempt 1: fails → wait 5 seconds
Attempt 2: fails → wait 25 seconds  (5 × 5)
Attempt 3: fails → wait 125 seconds (25 × 5)
Attempt 4: give up → send to Dead Letter Queue
```

The formula: `delay = base_delay × (multiplier ^ attempt)`

We also add a small random **jitter** so that many failing jobs don't all retry at the exact same moment (thundering herd).

In [ ]:
# Exponential backoff with jitter

def calculate_backoff(attempt: int, base_delay: float = 5.0, 
                      multiplier: float = 5.0, max_delay: float = 300.0) -> float:
    """Calculate retry delay with exponential backoff + jitter.
    
    Args:
        attempt: which retry this is (0-based)
        base_delay: initial delay in seconds
        multiplier: how much to multiply each time
        max_delay: cap so delays don't grow forever
    """
    delay = min(base_delay * (multiplier ** attempt), max_delay)
    # Add random jitter: ±25% of the delay
    jitter = delay * random.uniform(-0.25, 0.25)
    return max(0, delay + jitter)

# Show the backoff schedule
print("📊 Exponential Backoff Schedule")
print("=" * 50)
for attempt in range(6):
    delay = calculate_backoff(attempt)
    base = 5.0 * (5.0 ** attempt)
    capped = min(base, 300)
    print(f"  Attempt {attempt}: ~{capped:.0f}s base → {delay:.1f}s with jitter")

print()
print("💡 Each retry waits 5× longer, capped at 5 minutes.")
print("   Jitter prevents all retries from hitting at the same time.")

In [ ]:
# Full worker simulation with retries

MAX_RETRIES = 3
WORKER_LOG = []  # collect events for display

# Simulated task handlers — some succeed, some fail
def handle_send_email(params: dict) -> dict:
    """Simulated email sender — fails 40% of the time."""
    if random.random() < 0.4:
        raise Exception("SMTP connection timeout")
    return {"status": "sent", "message_id": f"msg_{uuid.uuid4().hex[:8]}"}

def handle_generate_report(params: dict) -> dict:
    """Simulated report generator — always succeeds."""
    time.sleep(0.01)  # simulate work
    return {"status": "generated", "pages": random.randint(5, 50)}

HANDLERS = {
    "send_email": handle_send_email,
    "generate_report": handle_generate_report,
}

def worker_loop(worker_id: str, max_jobs: int = 10):
    """Simulate a worker processing jobs from the queue."""
    processed = 0
    
    while processed < max_jobs:
        job = take_job(worker_id)
        if not job:
            break
        
        exec_id = job["execution_id"]
        task_id = job["task_id"]
        attempt = job.get("attempt", 0)
        handler = HANDLERS.get(task_id)
        
        if not handler:
            WORKER_LOG.append(f"  [{worker_id}] ❓ {exec_id}: unknown task '{task_id}'")
            ack_job(job)
            continue
        
        try:
            result = handler(job["parameters"])
            ack_job(job)
            WORKER_LOG.append(f"  [{worker_id}] ✅ {exec_id}: {task_id} succeeded (attempt {attempt})")
            processed += 1
            
        except Exception as e:
            ack_job(job)  # remove from processing set
            
            if attempt < MAX_RETRIES:
                # Retry with exponential backoff
                delay = calculate_backoff(attempt)
                retry_at = time.time() + delay
                
                # Re-enqueue with incremented attempt count
                retry_member = json.dumps({
                    "execution_id": exec_id,
                    "task_id": task_id,
                    "parameters": job["parameters"],
                    "attempt": attempt + 1
                })
                r.zadd(QUEUE_KEY, {retry_member: retry_at})
                WORKER_LOG.append(
                    f"  [{worker_id}] 🔄 {exec_id}: {task_id} failed (attempt {attempt}), "
                    f"retry in {delay:.0f}s — {e}"
                )
            else:
                # Max retries exceeded — this goes to Dead Letter Queue
                WORKER_LOG.append(
                    f"  [{worker_id}] ☠️  {exec_id}: {task_id} PERMANENTLY FAILED "
                    f"after {attempt + 1} attempts — {e}"
                )
            processed += 1

print("✅ Worker loop defined with retry logic")

In [ ]:
# Run the simulation: enqueue jobs, then process with multiple workers

r.delete(QUEUE_KEY, PROCESSING_KEY)
WORKER_LOG.clear()

# Enqueue 15 email jobs (some will fail and need retries)
now = time.time()
for i in range(15):
    enqueue(f"email-{i+1:02d}", "send_email", {"to": f"user{i+1}@example.com"})

print(f"📥 Enqueued 15 jobs")
print()

# Run 3 workers in parallel
threads = []
for w in range(3):
    t = threading.Thread(target=worker_loop, args=(f"worker-{w+1}", 10))
    threads.append(t)
    t.start()

for t in threads:
    t.join()

print("📋 Worker Activity Log:")
print("=" * 80)
for entry in WORKER_LOG:
    print(entry)

# Count outcomes
succeeded = sum(1 for e in WORKER_LOG if "✅" in e)
retried = sum(1 for e in WORKER_LOG if "🔄" in e)
failed = sum(1 for e in WORKER_LOG if "☠️" in e)

print()
print(f"📊 Results: {succeeded} succeeded, {retried} retried, {failed} permanently failed")
remaining = r.zcard(QUEUE_KEY)
if remaining:
    print(f"   {remaining} jobs waiting for retry in the queue")

## 🔒 Idempotency: Why At-Least-Once Needs Extra Care

Our visibility timeout guarantees **at-least-once** delivery — a job will execute at least once, possibly more (if the worker was slow but not dead).

This means your task code **must be idempotent**: running it twice should have the same effect as running it once.

### Bad (not idempotent):
```python
# Transfers money AGAIN on retry → user loses money twice!
def transfer_money(from_acct, to_acct, amount):
    db.execute("UPDATE accounts SET balance = balance - %s WHERE id = %s", (amount, from_acct))
    db.execute("UPDATE accounts SET balance = balance + %s WHERE id = %s", (amount, to_acct))
```

### Good (idempotent with idempotency key):
```python
def transfer_money(transaction_id, from_acct, to_acct, amount):
    # Check if this exact transfer already happened
    existing = db.query("SELECT 1 FROM transfers WHERE id = %s", (transaction_id,))
    if existing:
        return  # already processed, skip
    
    db.execute("INSERT INTO transfers (id, ...) VALUES (%s, ...)", (transaction_id, ...))
    db.execute("UPDATE accounts SET balance = balance - %s WHERE id = %s", (amount, from_acct))
    db.execute("UPDATE accounts SET balance = balance + %s WHERE id = %s", (amount, to_acct))
```

In [ ]:
# Demo: idempotency with a deduplication check

DEDUP_KEY = "worker_demo:dedup"
r.delete(DEDUP_KEY)

execution_counter = {"emails_sent": 0}

def send_email_NOT_idempotent(execution_id: str, params: dict):
    """BAD: sends the email every time, even on retry."""
    execution_counter["emails_sent"] += 1
    return f"Email #{execution_counter['emails_sent']} sent to {params['to']}"

def send_email_idempotent(execution_id: str, params: dict):
    """GOOD: checks if this execution already ran before sending."""
    # Check if we already processed this exact execution
    if r.sismember(DEDUP_KEY, execution_id):
        return f"Already processed {execution_id}, skipping"
    
    # Process the job
    execution_counter["emails_sent"] += 1
    result = f"Email #{execution_counter['emails_sent']} sent to {params['to']}"
    
    # Mark as processed (with TTL so the set doesn't grow forever)
    r.sadd(DEDUP_KEY, execution_id)
    r.expire(DEDUP_KEY, 86400)  # keep for 24 hours
    
    return result

# Simulate the same execution arriving 3 times (visibility timeout race)
params = {"to": "alice@example.com"}
exec_id = "exec-abc-123"

print("❌ Without idempotency (sends duplicate emails):")
execution_counter["emails_sent"] = 0
for i in range(3):
    result = send_email_NOT_idempotent(exec_id, params)
    print(f"   Delivery {i+1}: {result}")

print()
print("✅ With idempotency (deduplicates):")
r.delete(DEDUP_KEY)
execution_counter["emails_sent"] = 0
for i in range(3):
    result = send_email_idempotent(exec_id, params)
    print(f"   Delivery {i+1}: {result}")

print()
print("💡 The idempotent version only sends 1 email even though it was called 3 times.")
print("   This is critical for at-least-once delivery systems.")

## 💓 Worker Heartbeats: Extending Visibility

Some jobs take longer than the default visibility timeout. A 5-minute report generation shouldn't be killed after 30 seconds.

The solution: **heartbeats**. While processing, the worker periodically extends its timeout:

```
Worker takes job (timeout = 30s)
  │
  ├─ 15s: heartbeat → extend to 45s
  ├─ 30s: heartbeat → extend to 60s
  ├─ 45s: heartbeat → extend to 75s
  │
  └─ 50s: job completes → ACK ✅
```

If the worker crashes, heartbeats stop, and the visibility timeout expires normally.

In [ ]:
# Heartbeat implementation

def extend_visibility(job: dict, extension_seconds: int = 30):
    """Extend the visibility timeout for a job being processed.
    
    Call this periodically while processing a long-running job.
    """
    member = json.dumps(job)
    new_timeout = time.time() + extension_seconds
    # Update the score in the processing set
    r.zadd(PROCESSING_KEY, {member: new_timeout})

def process_with_heartbeat(worker_id: str, job: dict, 
                           work_fn, heartbeat_interval: int = 15):
    """Process a job with periodic heartbeats to extend visibility.
    
    Runs the work function in a thread while the main thread
    sends heartbeats to keep the visibility timeout alive.
    """
    result = {"value": None, "error": None}
    done = threading.Event()
    
    def do_work():
        try:
            result["value"] = work_fn()
        except Exception as e:
            result["error"] = str(e)
        finally:
            done.set()
    
    # Start the work in a background thread
    work_thread = threading.Thread(target=do_work)
    work_thread.start()
    
    # Send heartbeats until work completes
    heartbeat_count = 0
    while not done.wait(timeout=heartbeat_interval):
        extend_visibility(job, extension_seconds=VISIBILITY_TIMEOUT)
        heartbeat_count += 1
    
    work_thread.join()
    return result["value"], result["error"], heartbeat_count

# Demo: process a "long" job with heartbeats
r.delete(QUEUE_KEY, PROCESSING_KEY)
enqueue("long-job-001", "generate_report", {"type": "annual"})

job = take_job("worker-heartbeat")
print(f"👷 Took job: {job['execution_id']}")
print(f"   Visibility timeout: {VISIBILITY_TIMEOUT}s")
print(f"   Simulating a 5-second job with 2-second heartbeat interval...")
print()

def slow_work():
    time.sleep(5)  # simulate slow work
    return {"pages": 120, "format": "pdf"}

value, error, heartbeats = process_with_heartbeat(
    "worker-heartbeat", job, slow_work, heartbeat_interval=2
)

if error:
    print(f"❌ Failed: {error}")
else:
    ack_job(job)
    print(f"✅ Completed: {value}")
    print(f"   Sent {heartbeats} heartbeats during processing")
    print()
    print("💡 Without heartbeats, this job would have timed out.")
    print("   Heartbeats keep the worker's claim alive for long-running jobs.")

## 🧹 Cleanup

In [ ]:
r = get_redis()
for key in r.keys("worker_demo:*"):
    r.delete(key)
print("🧹 Cleaned up Redis keys")

## 📚 Summary

### Key Takeaways

1. **Visibility timeouts** ensure no job is lost when a worker crashes — the job reappears after the timeout
2. **Exponential backoff + jitter** prevents retry storms and gives failing services time to recover
3. **At-least-once delivery** means jobs may run more than once — your tasks **must be idempotent**
4. **Heartbeats** extend visibility for long-running jobs without increasing the default timeout
5. **Deduplication** (idempotency keys) protects against duplicate execution

### Interview Tips

- Mention both **visible failures** (exceptions) and **invisible failures** (worker crash)
- Always discuss idempotency when you mention at-least-once delivery
- Heartbeats show senior-level thinking about edge cases

### Next Up

In **Notebook 3**, we'll build **cron-like recurring schedules** and implement a **dead letter queue** for permanently failed jobs.